In [2]:
from collections import defaultdict
def read_corpus():
    return [
        "low",
        "lower",
        "lowest",
        "new",
        "newer",
        "widest",
        "low",
        "lower",
        "new",
        "lowest"
    ]
def get_vocab(words):
    vocab = defaultdict(int)

    for word in words:
        chars = " ".join(list(word)) + " </w>"
        vocab[chars] += 1

    return vocab
def get_stats(vocab):
    pairs = defaultdict(int)

    for word, freq in vocab.items():
        symbols = word.split()

        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq

    return pairs
def merge_vocab(pair, vocab):
    new_vocab = {}

    bigram = " ".join(pair)
    replacement = "".join(pair)

    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]

    return new_vocab
def learn_bpe(vocab, num_merges):
    merges = []

    for i in range(num_merges):

        pairs = get_stats(vocab)

        if not pairs:
            break

        best = max(pairs, key=pairs.get)

        print(f"Merge {i+1}: {best}")

        vocab = merge_vocab(best, vocab)

        merges.append(best)

    return vocab, merges
def build_final_vocab(vocab):
    final_vocab = set()

    for word in vocab:
        for token in word.split():
            final_vocab.add(token)

    return sorted(final_vocab)
def encode(word, merges):

    tokens = list(word) + ["</w>"]

    for pair in merges:

        i = 0

        while i < len(tokens) - 1:

            if tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
                tokens[i:i+2] = ["".join(pair)]
            else:
                i += 1

    return tokens
def decode(tokens):

    word = ""

    for token in tokens:
        if token != "</w>":
            word += token

    return word.replace("</w>", "")
if __name__ == "__main__":

    words = read_corpus()

    print("Corpus")
    print(words)

    vocab = get_vocab(words)

    print("\nInitial Vocabulary")
    for word, freq in vocab.items():
        print(word, ":", freq)

    NUM_MERGES = 10

    vocab, merges = learn_bpe(vocab, NUM_MERGES)

    print("\nMerge Rules")
    for rule in merges:
        print(rule)

    final_vocab = build_final_vocab(vocab)

    print("\nFinal Vocabulary")
    print(final_vocab)

    print("\nVocabulary Size:", len(final_vocab))

    test_word = "lowest"

    encoded = encode(test_word, merges)

    print("\nTest Word:", test_word)
    print("Encoded:", encoded)

    decoded = decode(encoded)

    print("Decoded:", decoded)


Corpus
['low', 'lower', 'lowest', 'new', 'newer', 'widest', 'low', 'lower', 'new', 'lowest']

Initial Vocabulary
l o w </w> : 2
l o w e r </w> : 2
l o w e s t </w> : 2
n e w </w> : 2
n e w e r </w> : 1
w i d e s t </w> : 1
Merge 1: ('l', 'o')
Merge 2: ('lo', 'w')
Merge 3: ('low', 'e')
Merge 4: ('r', '</w>')
Merge 5: ('s', 't')
Merge 6: ('st', '</w>')
Merge 7: ('n', 'e')
Merge 8: ('ne', 'w')
Merge 9: ('low', '</w>')
Merge 10: ('lowe', 'r</w>')

Merge Rules
('l', 'o')
('lo', 'w')
('low', 'e')
('r', '</w>')
('s', 't')
('st', '</w>')
('n', 'e')
('ne', 'w')
('low', '</w>')
('lowe', 'r</w>')

Final Vocabulary
['</w>', 'd', 'e', 'i', 'low</w>', 'lowe', 'lower</w>', 'new', 'r</w>', 'st</w>', 'w']

Vocabulary Size: 11

Test Word: lowest
Encoded: ['lowe', 'st</w>']
Decoded: lowest
